# SysCraft Arena MQTT Entity Loop

Publishes moving demo entities into `syscraft/arena/{domain}/{assetType}/{entityId}` so the Arena map and asset palette can render live MQTT data.

## 1. Install Dependency

Run this once in your Jupyter environment if `paho-mqtt` is not already installed.

In [ ]:
# %pip install -r ../scripts/requirements.txt

## 2. Configure Broker

Important for the current EMQX authorization setup: keep `MQTT_CLIENT_ID = "chris"` unless you add another client ID rule in EMQX.

In [ ]:
import getpass
import os

MQTT_HOST = os.getenv("SYSCRAFT_MQTT_HOST", "mqtt.conceptio.cloud")
MQTT_PORT = int(os.getenv("SYSCRAFT_MQTT_PORT", "1883"))
MQTT_USERNAME = os.getenv("SYSCRAFT_MQTT_USERNAME", "chris")
MQTT_CLIENT_ID = os.getenv("SYSCRAFT_MQTT_CLIENT_ID", "chris")
MQTT_ROOT_TOPIC = os.getenv("SYSCRAFT_MQTT_ROOT_TOPIC", "syscraft/arena")
MQTT_INTERVAL_SECONDS = float(os.getenv("SYSCRAFT_MQTT_INTERVAL", "1.0"))
MQTT_PASSWORD = os.getenv("SYSCRAFT_MQTT_PASSWORD") or getpass.getpass("MQTT password: ")

print(f"Broker: {MQTT_HOST}:{MQTT_PORT}")
print(f"Client ID: {MQTT_CLIENT_ID}")
print(f"Publishing under: {MQTT_ROOT_TOPIC}/...")

## 3. Create Publisher Controls

In [ ]:
import json
import pathlib
import sys
import threading
import time

def find_project_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "mqtt_entity_loop.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find scripts/mqtt_entity_loop.py. "
        "Open this notebook from the skycraft_multidomain_arena project folder, "
        "or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root(pathlib.Path.cwd().resolve())
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import paho.mqtt.client as mqtt
import mqtt_entity_loop as arena_mqtt

publisher_state = {
    "client": None,
    "thread": None,
    "stop": threading.Event(),
    "tick": 0,
}


def _publish_loop():
    client = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=MQTT_CLIENT_ID,
        protocol=mqtt.MQTTv5,
    )
    client.username_pw_set(MQTT_USERNAME, MQTT_PASSWORD)

    def on_connect(_client, _userdata, _flags, reason_code, _properties):
        print(f"Connected: {reason_code}")

    def on_disconnect(_client, _userdata, _flags, reason_code, _properties):
        if not publisher_state["stop"].is_set():
            print(f"Disconnected: {reason_code}")

    client.on_connect = on_connect
    client.on_disconnect = on_disconnect
    client.connect(MQTT_HOST, MQTT_PORT, 60)
    client.loop_start()
    publisher_state["client"] = client
    started_at = time.time()

    try:
        while not publisher_state["stop"].is_set():
            publisher_state["tick"] += 1
            tick = publisher_state["tick"]
            for entity in arena_mqtt.ENTITY_SEEDS:
                topic = arena_mqtt.topic_for(MQTT_ROOT_TOPIC, entity)
                payload = arena_mqtt.payload_for(entity, MQTT_ROOT_TOPIC, tick, started_at)
                client.publish(topic, json.dumps(payload, separators=(",", ":")), qos=0, retain=False)

            if tick == 1 or tick % 10 == 0:
                print(f"tick={tick} published={len(arena_mqtt.ENTITY_SEEDS)}")

            time.sleep(MQTT_INTERVAL_SECONDS)
    finally:
        client.loop_stop()
        client.disconnect()
        publisher_state["client"] = None
        print("Publisher stopped.")


def start_publisher():
    thread = publisher_state.get("thread")
    if thread and thread.is_alive():
        print("Publisher already running.")
        return

    publisher_state["stop"].clear()
    publisher_state["thread"] = threading.Thread(target=_publish_loop, daemon=True)
    publisher_state["thread"].start()
    print("Publisher starting...")


def stop_publisher():
    publisher_state["stop"].set()
    thread = publisher_state.get("thread")
    if thread:
        thread.join(timeout=5)
    print("Stop requested.")

## 4. Start Publishing

Keep the Arena UI connected with:

- Protocol: `ws`
- Port: `8083`
- Path: `/mqtt`
- Client ID: `chris`
- Root topic: `syscraft/arena`

In [ ]:
start_publisher()

## 5. Stop Publishing

In [ ]:
stop_publisher()